# Reproducible Development-Pilot Sampling

## Purpose

This notebook creates a balanced and reproducible development sample
for the provisional Benign-versus-DoS experiment.

Unlike the earlier engineering draft, this procedure operates only after
the data-audit and data-preparation policies have been defined.

## Inputs

The notebook reads:

1. the immutable provider-supplied `NF-UNSW-NB15-v3.csv` file; and
2. `configs/feature_policy_provisional.csv`, which defines the 48
   provisional model-input fields and seven excluded fields.

## Sampling unit

The sampling unit is a unique 48-field model-input profile rather than
an individual source-row occurrence.

This prevents identical model inputs from receiving multiple sampling
opportunities merely because the same observable profile occurs more
than once in the source dataset.

## Development-sample design

The provisional sample contains:

- 100 unique Benign model-input profiles;
- 100 unique DoS model-input profiles; and
- 200 profiles in total.

Sampling uses the fixed random seed `742`.

## Non-finite values

Records are not excluded merely because
`SRC_TO_DST_SECOND_BYTES` or `DST_TO_SRC_SECOND_BYTES` contains a missing
or infinite value.

These states will later be represented explicitly and equivalently in
both the structured-value and deterministic-text conditions.

## Ground-truth separation

`Label` and `Attack` are retained only in a private evaluation table.
They must not appear in any prompt or model-facing input artifact.

## Status

This is a development smoke-test sample, not the final evaluation set.
The final dataset design remains subject to clarification of the
measured-benign-trace requirement.

In [1]:
# ---------------------------------------------------------------------
# Import tools used for paths, hashing, random sampling and tabular data
# ---------------------------------------------------------------------

from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Fixed development-sampling configuration
# ---------------------------------------------------------------------

# The same seed reproduces the same pseudo-random priorities when the
# source CSV, feature policy, code and chunk size remain unchanged.
RANDOM_SEED = 742

# The development pilot currently studies binary classification.
TARGET_CATEGORIES = ["Benign", "DoS"]

# Each class contributes the same number of distinct model profiles.
PROFILES_PER_CATEGORY = 100

# Chunking avoids loading the complete 550 MiB source file into memory.
CHUNK_SIZE = 100_000

sampling_configuration = pd.Series(
    {
        "random_seed": RANDOM_SEED,
        "target_categories": ", ".join(
            TARGET_CATEGORIES
        ),
        "profiles_per_category": (
            PROFILES_PER_CATEGORY
        ),
        "total_requested_profiles": (
            len(TARGET_CATEGORIES)
            * PROFILES_PER_CATEGORY
        ),
        "chunk_size": CHUNK_SIZE,
        "sampling_unit": (
            "unique_48_field_model_input_profile"
        ),
        "python_version": sys.version.split()[0],
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "platform": platform.platform(),
    },
    name="value",
)

sampling_configuration

random_seed                                                 742
target_categories                                   Benign, DoS
profiles_per_category                                       100
total_requested_profiles                                    200
chunk_size                                               100000
sampling_unit               unique_48_field_model_input_profile
python_version                                          3.11.14
pandas_version                                            3.0.5
numpy_version                                             2.4.6
platform                           macOS-15.7.3-arm64-arm-64bit
Name: value, dtype: object

In [2]:
# ---------------------------------------------------------------------
# Locate and validate the source dataset and feature-policy file
# ---------------------------------------------------------------------

# The notebook lives inside notebooks/, so its parent is the project root.
PROJECT_ROOT = Path.cwd().resolve().parent

SOURCE_CSV = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nf_unsw_nb15_v3"
    / "NF-UNSW-NB15-v3.csv"
)

FEATURE_POLICY_CSV = (
    PROJECT_ROOT
    / "configs"
    / "feature_policy_provisional.csv"
)

INTERIM_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
)

OUTPUT_SAMPLE_CSV = (
    INTERIM_DATA_DIR
    / "pilot_benign_dos_unique_profiles_n200.csv"
)

OUTPUT_MANIFEST_JSON = (
    INTERIM_DATA_DIR
    / "pilot_benign_dos_unique_profiles_n200_manifest.json"
)

assert SOURCE_CSV.exists(), (
    f"Source CSV not found:\n{SOURCE_CSV}"
)

assert FEATURE_POLICY_CSV.exists(), (
    f"Feature-policy file not found:\n"
    f"{FEATURE_POLICY_CSV}"
)

# This generated-data directory is ignored by Git.
INTERIM_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

path_check = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "source_csv_exists": SOURCE_CSV.exists(),
        "feature_policy_exists": (
            FEATURE_POLICY_CSV.exists()
        ),
        "output_sample": str(OUTPUT_SAMPLE_CSV),
        "output_manifest": str(
            OUTPUT_MANIFEST_JSON
        ),
    },
    name="value",
)

path_check

project_root                 /Users/ruiwang/Developer/compsci742-rui-pilot
source_csv_exists                                                     True
feature_policy_exists                                                 True
output_sample            /Users/ruiwang/Developer/compsci742-rui-pilot/...
output_manifest          /Users/ruiwang/Developer/compsci742-rui-pilot/...
Name: value, dtype: object

In [3]:
# ---------------------------------------------------------------------
# Load the data-preparation policy instead of redefining features here
# ---------------------------------------------------------------------

feature_policy = pd.read_csv(
    FEATURE_POLICY_CSV
)

# CSV stores the Boolean decision as True/False. This normalisation also
# handles the possibility that another CSV reader returns text strings.
core_input_flag = (
    feature_policy["provisional_core_input"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        {
            "true": True,
            "false": False,
        }
    )
)

assert core_input_flag.notna().all(), (
    "The provisional_core_input column contains an "
    "unrecognised value."
)

model_input_columns = (
    feature_policy.loc[
        core_input_flag,
        "column_name",
    ]
    .tolist()
)

excluded_columns = (
    feature_policy.loc[
        ~core_input_flag,
        "column_name",
    ]
    .tolist()
)

assert len(feature_policy) == 55
assert len(model_input_columns) == 48
assert len(excluded_columns) == 7
assert "Label" not in model_input_columns
assert "Attack" not in model_input_columns

policy_check = pd.Series(
    {
        "policy_rows": len(feature_policy),
        "model_input_columns": len(
            model_input_columns
        ),
        "excluded_columns": len(
            excluded_columns
        ),
        "label_excluded": (
            "Label" not in model_input_columns
        ),
        "attack_excluded": (
            "Attack" not in model_input_columns
        ),
    },
    name="value",
)

policy_check

policy_rows              55
model_input_columns      48
excluded_columns          7
label_excluded         True
attack_excluded        True
Name: value, dtype: object

## Deterministic profile-priority sampling

Each unique 48-field model-input profile is converted into a stable
64-bit hash. Records with identical model-facing values therefore receive
the same profile hash.

The profile hash and fixed seed are then mixed into a deterministic
sampling priority. Consequently:

- each distinct profile receives exactly one effective sampling priority;
- duplicate source occurrences do not gain extra sampling opportunities;
- chunk order does not change the selected profiles; and
- rerunning the procedure with the same data, policy, seed and software
  environment reproduces the sample.

For each category, the 100 unique profiles with the smallest priorities
are retained.

In [5]:
# ---------------------------------------------------------------------
# Convert stable profile hashes into seeded sampling priorities
# ---------------------------------------------------------------------

def create_seeded_priorities(profile_hashes, seed):
    """
    Mix 64-bit profile hashes with a fixed seed.

    The calculations intentionally use unsigned-integer overflow. This
    produces a well-distributed deterministic priority without relying
    on the number or order of duplicate source rows.

    Parameters
    ----------
    profile_hashes:
        NumPy array containing one uint64 hash per record.
    seed:
        Fixed integer used to reproduce the same priorities.

    Returns
    -------
    NumPy array of deterministic uint64 sampling priorities.
    """
    values = profile_hashes.astype(
        np.uint64,
        copy=True,
    )

    # Add the experiment seed before applying the SplitMix64-style
    # integer-mixing operations.
    values += np.uint64(seed)

    # Unsigned overflow is deliberate in this hash-mixing function.
    with np.errstate(over="ignore"):
        values ^= values >> np.uint64(30)
        values *= np.uint64(
            0xBF58476D1CE4E5B9
        )

        values ^= values >> np.uint64(27)
        values *= np.uint64(
            0x94D049BB133111EB
        )

        values ^= values >> np.uint64(31)

    return values

In [6]:
# ---------------------------------------------------------------------
# Scan the complete source CSV and retain the best unique profiles
# ---------------------------------------------------------------------

# Each category starts with an empty reservoir.
# A reservoir is the small table of currently selected candidates.
profile_reservoirs = {
    category: pd.DataFrame()
    for category in TARGET_CATEGORIES
}

rows_processed = 0

target_source_rows_seen = {
    category: 0
    for category in TARGET_CATEGORIES
}


for chunk_number, chunk in enumerate(
    pd.read_csv(
        SOURCE_CSV,
        chunksize=CHUNK_SIZE,
    ),
    start=1,
):
    # Create a stable zero-based reference to the original CSV data row.
    # This is retained privately for reproducibility and must not be sent
    # to a model.
    chunk["source_row_id"] = (
        rows_processed
        + np.arange(
            len(chunk),
            dtype=np.int64,
        )
    )

    # Restrict the current experiment to Benign and DoS.
    target_chunk = chunk[
        chunk["Attack"].isin(
            TARGET_CATEGORIES
        )
    ].copy()

    # Generate the model-input profile fingerprint from only the 48
    # fields defined by the feature-policy file.
    target_chunk["_model_profile_hash"] = (
        pd.util.hash_pandas_object(
            target_chunk[
                model_input_columns
            ],
            index=False,
        ).to_numpy(dtype=np.uint64)
    )

    # Duplicate model profiles receive the same deterministic priority.
    target_chunk["_sampling_priority"] = (
        create_seeded_priorities(
            target_chunk[
                "_model_profile_hash"
            ].to_numpy(dtype=np.uint64),
            RANDOM_SEED,
        )
    )

    for category in TARGET_CATEGORIES:
        category_candidates = target_chunk[
            target_chunk["Attack"] == category
        ].copy()

        target_source_rows_seen[
            category
        ] += len(category_candidates)

        # Combine this chunk's candidates with the small reservoir retained
        # from all earlier chunks.
        combined_candidates = pd.concat(
            [
                profile_reservoirs[category],
                category_candidates,
            ],
            ignore_index=True,
        )

        # Priority determines which profiles are selected.
        # source_row_id breaks ties between repeated occurrences of the
        # same profile, retaining the earliest representative source row.
        combined_candidates = (
            combined_candidates
            .sort_values(
                [
                    "_sampling_priority",
                    "source_row_id",
                ],
                kind="mergesort",
            )
            .drop_duplicates(
                subset="_model_profile_hash",
                keep="first",
            )
            .head(PROFILES_PER_CATEGORY)
            .copy()
        )

        profile_reservoirs[
            category
        ] = combined_candidates

    rows_processed += len(chunk)

    if chunk_number == 1 or chunk_number % 5 == 0:
        print(
            f"Processed chunk {chunk_number:>2}: "
            f"{rows_processed:,} cumulative source rows"
        )


print("\nUnique-profile sampling scan completed.")
print(f"Total source rows scanned: {rows_processed:,}")

for category in TARGET_CATEGORIES:
    print(
        f"{category} source rows considered: "
        f"{target_source_rows_seen[category]:,}"
    )

    print(
        f"{category} unique profiles retained: "
        f"{len(profile_reservoirs[category]):,}"
    )

Processed chunk  1: 100,000 cumulative source rows
Processed chunk  5: 500,000 cumulative source rows
Processed chunk 10: 1,000,000 cumulative source rows
Processed chunk 15: 1,500,000 cumulative source rows
Processed chunk 20: 2,000,000 cumulative source rows

Unique-profile sampling scan completed.
Total source rows scanned: 2,365,424
Benign source rows considered: 2,237,731
Benign unique profiles retained: 100
DoS source rows considered: 5,980
DoS unique profiles retained: 100


In [7]:
# ---------------------------------------------------------------------
# Assemble and validate the balanced development sample
# ---------------------------------------------------------------------

pilot_sample = pd.concat(
    [
        profile_reservoirs[category]
        for category in TARGET_CATEGORIES
    ],
    ignore_index=True,
)

# Shuffle the final order reproducibly so that records are not grouped
# by ground-truth category.
pilot_sample = (
    pilot_sample
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

# Create a neutral experiment identifier.
pilot_sample.insert(
    0,
    "sample_id",
    [
        f"pilot_{position:03d}"
        for position in range(
            1,
            len(pilot_sample) + 1,
        )
    ],
)

# Convert the uint64 hash into a fixed hexadecimal identifier.
# This avoids precision loss if the manifest is later read by software
# that cannot represent every 64-bit unsigned integer exactly.
pilot_sample.insert(
    2,
    "model_profile_id",
    pilot_sample[
        "_model_profile_hash"
    ].map(
        lambda value: f"{int(value):016x}"
    ),
)


# ---------------------------------------------------------------------
# Validate size, balance, uniqueness and ground-truth separation
# ---------------------------------------------------------------------

expected_sample_size = (
    len(TARGET_CATEGORIES)
    * PROFILES_PER_CATEGORY
)

assert len(pilot_sample) == expected_sample_size

assert pilot_sample[
    "sample_id"
].is_unique

assert pilot_sample[
    "source_row_id"
].is_unique

assert pilot_sample[
    "model_profile_id"
].is_unique, (
    "Duplicate model-input profiles entered the sample."
)

assert (
    pilot_sample["Attack"]
    .value_counts()
    .reindex(TARGET_CATEGORIES)
    == PROFILES_PER_CATEGORY
).all(), (
    "The sample is not balanced."
)

assert "Label" not in model_input_columns
assert "Attack" not in model_input_columns
assert "sample_id" not in model_input_columns
assert "source_row_id" not in model_input_columns
assert "model_profile_id" not in model_input_columns

sample_validation = pd.Series(
    {
        "sample_records": len(pilot_sample),
        "unique_sample_ids": (
            pilot_sample["sample_id"].nunique()
        ),
        "unique_source_rows": (
            pilot_sample["source_row_id"].nunique()
        ),
        "unique_model_profiles": (
            pilot_sample[
                "model_profile_id"
            ].nunique()
        ),
        "benign_records": int(
            (pilot_sample["Attack"] == "Benign").sum()
        ),
        "dos_records": int(
            (pilot_sample["Attack"] == "DoS").sum()
        ),
        "ground_truth_in_model_input": bool(
            {"Label", "Attack"}
            .intersection(model_input_columns)
        ),
    },
    name="value",
)

sample_validation

sample_records                   200
unique_sample_ids                200
unique_source_rows               200
unique_model_profiles            200
benign_records                   100
dos_records                      100
ground_truth_in_model_input    False
Name: value, dtype: object

## Selected-sample composition checks

Structural validation confirms that the sample is balanced and contains
200 unique model-input profiles.

The following checks inspect, without modifying the sample:

- missing and infinite directional-rate values;
- protocol-code coverage; and
- temporal coverage of the representative source records.

These checks are descriptive. The sample will not be repeatedly redrawn
merely to obtain a more desirable-looking composition.

In [8]:
# ---------------------------------------------------------------------
# Inspect non-finite values in the selected sample
# ---------------------------------------------------------------------

src_rate = pilot_sample[
    "SRC_TO_DST_SECOND_BYTES"
].to_numpy()

dst_rate = pilot_sample[
    "DST_TO_SRC_SECOND_BYTES"
].to_numpy()

sample_quality_flags = pd.DataFrame(
    {
        "Attack": pilot_sample["Attack"],
        "src_rate_missing": pd.isna(src_rate),
        "src_rate_infinite": np.isinf(src_rate),
        "dst_rate_missing": pd.isna(dst_rate),
        "dst_rate_infinite": np.isinf(dst_rate),
    }
)

sample_quality_flags["any_nonfinite_rate"] = (
    sample_quality_flags[
        [
            "src_rate_missing",
            "src_rate_infinite",
            "dst_rate_missing",
            "dst_rate_infinite",
        ]
    ].any(axis=1)
)

sample_quality_summary = (
    sample_quality_flags
    .groupby("Attack")
    .agg(
        sample_records=(
            "any_nonfinite_rate",
            "size",
        ),
        src_missing=(
            "src_rate_missing",
            "sum",
        ),
        src_infinite=(
            "src_rate_infinite",
            "sum",
        ),
        dst_missing=(
            "dst_rate_missing",
            "sum",
        ),
        dst_infinite=(
            "dst_rate_infinite",
            "sum",
        ),
        records_with_any_nonfinite=(
            "any_nonfinite_rate",
            "sum",
        ),
    )
)

sample_quality_summary[
    "nonfinite_percentage"
] = (
    100
    * sample_quality_summary[
        "records_with_any_nonfinite"
    ]
    / sample_quality_summary[
        "sample_records"
    ]
)

sample_quality_summary

,sample_records,src_missing,src_infinite,dst_missing,dst_infinite,records_with_any_nonfinite,nonfinite_percentage
Attack,,,,,,,
Benign,100,0,5,0,5,5,5.0
DoS,100,19,0,0,19,19,19.0


In [9]:
# ---------------------------------------------------------------------
# Inspect protocol-code composition within each category
# ---------------------------------------------------------------------

protocol_count_table = pd.crosstab(
    pilot_sample["PROTOCOL"],
    pilot_sample["Attack"],
)

protocol_percentage_table = (
    pd.crosstab(
        pilot_sample["PROTOCOL"],
        pilot_sample["Attack"],
        normalize="columns",
    )
    * 100
)

print("Selected records by protocol code:")
display(protocol_count_table)

print("\nPercentage within each category:")
display(
    protocol_percentage_table.round(2)
)

Selected records by protocol code:


Attack,Benign,DoS
PROTOCOL,,
6,85,81
17,15,19



Percentage within each category:


Attack,Benign,DoS
PROTOCOL,,
6,85.0,81.0
17,15.0,19.0


In [10]:
# ---------------------------------------------------------------------
# Inspect temporal coverage of representative source records
# ---------------------------------------------------------------------

sample_time_audit = pilot_sample[
    [
        "Attack",
        "FLOW_START_MILLISECONDS",
        "FLOW_END_MILLISECONDS",
    ]
].copy()

sample_time_audit["flow_start_utc"] = pd.to_datetime(
    sample_time_audit[
        "FLOW_START_MILLISECONDS"
    ],
    unit="ms",
    utc=True,
)

sample_time_audit["flow_end_utc"] = pd.to_datetime(
    sample_time_audit[
        "FLOW_END_MILLISECONDS"
    ],
    unit="ms",
    utc=True,
)

temporal_coverage_summary = (
    sample_time_audit
    .groupby("Attack")
    .agg(
        earliest_start_utc=(
            "flow_start_utc",
            "min",
        ),
        latest_start_utc=(
            "flow_start_utc",
            "max",
        ),
        earliest_end_utc=(
            "flow_end_utc",
            "min",
        ),
        latest_end_utc=(
            "flow_end_utc",
            "max",
        ),
        distinct_start_dates=(
            "flow_start_utc",
            lambda values: (
                values.dt.date.nunique()
            ),
        ),
    )
)

temporal_coverage_summary

,earliest_start_utc,latest_start_utc,earliest_end_utc,latest_end_utc,distinct_start_dates
Attack,,,,,
Benign,2015-01-22 12:13:49.943000+00:00,2015-02-18 12:14:21.416000+00:00,2015-01-22 12:13:49.945000+00:00,2015-02-18 12:14:21.420000+00:00,2
DoS,2015-01-22 11:59:14.894000+00:00,2015-02-18 12:14:56.984000+00:00,2015-01-22 11:59:14.894000+00:00,2015-02-18 12:14:57.608000+00:00,2


## Population-date context

The selected Benign and DoS profiles span the earliest and latest dates
visible in the current sample, but each category contains only two
distinct start dates.

A population-level date count is therefore required to determine whether
this reflects the source dataset's collection structure or an accidental
concentration in the selected sample.

In [11]:
# ---------------------------------------------------------------------
# Compare sample date coverage with the complete target population
# ---------------------------------------------------------------------

from collections import Counter

population_date_counts = {
    category: Counter()
    for category in TARGET_CATEGORIES
}

DATE_AUDIT_COLUMNS = [
    "Attack",
    "FLOW_START_MILLISECONDS",
]

date_rows_processed = 0


for chunk_number, chunk in enumerate(
    pd.read_csv(
        SOURCE_CSV,
        usecols=DATE_AUDIT_COLUMNS,
        chunksize=CHUNK_SIZE,
    ),
    start=1,
):
    target_chunk = chunk[
        chunk["Attack"].isin(
            TARGET_CATEGORIES
        )
    ].copy()

    # Convert milliseconds since the Unix epoch to a UTC calendar date.
    target_chunk["flow_start_date"] = (
        pd.to_datetime(
            target_chunk[
                "FLOW_START_MILLISECONDS"
            ],
            unit="ms",
            utc=True,
        )
        .dt.strftime("%Y-%m-%d")
    )

    for category in TARGET_CATEGORIES:
        category_dates = target_chunk.loc[
            target_chunk["Attack"] == category,
            "flow_start_date",
        ]

        population_date_counts[
            category
        ].update(
            category_dates.value_counts().to_dict()
        )

    date_rows_processed += len(chunk)


population_date_rows = []

for category in TARGET_CATEGORIES:
    total_category_records = sum(
        population_date_counts[
            category
        ].values()
    )

    for date, count in sorted(
        population_date_counts[
            category
        ].items()
    ):
        population_date_rows.append(
            {
                "Attack": category,
                "flow_start_date": date,
                "record_count": count,
                "percentage_within_category": (
                    100
                    * count
                    / total_category_records
                ),
            }
        )

population_date_distribution = pd.DataFrame(
    population_date_rows
)

population_date_distribution

,Attack,flow_start_date,record_count,percentage_within_category
0,Benign,2015-01-22,1078112,48.178803
1,Benign,2015-01-23,39778,1.777604
2,Benign,2015-02-18,1119841,50.043593
3,DoS,2015-01-22,624,10.434783
4,DoS,2015-02-18,5356,89.565217


## Temporal-context finding

The provisional Benign and DoS populations have different collection-date
distributions.

Benign records occur on three start dates:

- 2015-01-22: 48.18%;
- 2015-01-23: 1.78%; and
- 2015-02-18: 50.04%.

DoS records occur on two start dates:

- 2015-01-22: 10.43%; and
- 2015-02-18: 89.57%.

The selected development sample covers 2015-01-22 and 2015-02-18 but
contains no Benign profile from the relatively rare 2015-01-23 capture
date.

The sample is retained because it was generated using the pre-specified
seed and unique-profile procedure. It will not be redrawn to obtain a
more desirable composition.

However, the class-dependent date distribution is a potential collection
context confound. The final protocol should consider date matching,
date-stratified sampling or a temporal sensitivity analysis.

In [12]:
# ---------------------------------------------------------------------
# Separate model-facing features from private evaluation information
# ---------------------------------------------------------------------

# The feature table contains neutral identifiers plus the 48 fields
# defined by the feature-policy file.
pilot_features = pilot_sample[
    [
        "sample_id",
        "model_profile_id",
        *model_input_columns,
    ]
].copy()

# Ground truth and provenance are kept in a physically separate table.
# IP addresses are not copied into either generated output.
pilot_ground_truth = pilot_sample[
    [
        "sample_id",
        "model_profile_id",
        "source_row_id",
        "Label",
        "Attack",
        "FLOW_START_MILLISECONDS",
        "FLOW_END_MILLISECONDS",
    ]
].copy()


# ---------------------------------------------------------------------
# Safety checks
# ---------------------------------------------------------------------

assert pilot_features.shape == (
    200,
    50,
), (
    "Expected 200 rows and 50 columns: "
    "2 neutral identifiers plus 48 model fields."
)

assert pilot_ground_truth.shape == (
    200,
    7,
)

assert not {
    "Label",
    "Attack",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "source_row_id",
}.intersection(
    pilot_features.columns
), (
    "Private or excluded information entered the feature table."
)

assert pilot_features[
    "sample_id"
].tolist() == pilot_ground_truth[
    "sample_id"
].tolist()

assert pilot_features[
    "model_profile_id"
].tolist() == pilot_ground_truth[
    "model_profile_id"
].tolist()

separation_check = pd.Series(
    {
        "feature_rows": len(pilot_features),
        "feature_columns": len(
            pilot_features.columns
        ),
        "model_fields": len(
            model_input_columns
        ),
        "ground_truth_rows": len(
            pilot_ground_truth
        ),
        "ground_truth_columns": len(
            pilot_ground_truth.columns
        ),
        "label_in_feature_table": (
            "Label" in pilot_features.columns
        ),
        "attack_in_feature_table": (
            "Attack" in pilot_features.columns
        ),
        "ip_in_feature_table": bool(
            {
                "IPV4_SRC_ADDR",
                "IPV4_DST_ADDR",
            }.intersection(
                pilot_features.columns
            )
        ),
    },
    name="value",
)

separation_check

feature_rows                 200
feature_columns               50
model_fields                  48
ground_truth_rows            200
ground_truth_columns           7
label_in_feature_table     False
attack_in_feature_table    False
ip_in_feature_table        False
Name: value, dtype: object

In [13]:
# ---------------------------------------------------------------------
# Save the separated development artifacts and reproducibility manifest
# ---------------------------------------------------------------------

import hashlib


OUTPUT_FEATURES_CSV = (
    INTERIM_DATA_DIR
    / "pilot_benign_dos_n200_features.csv"
)

OUTPUT_GROUND_TRUTH_CSV = (
    INTERIM_DATA_DIR
    / "pilot_benign_dos_n200_ground_truth_private.csv"
)

OUTPUT_MANIFEST_JSON = (
    INTERIM_DATA_DIR
    / "pilot_benign_dos_n200_manifest.json"
)


# "NaN" is written explicitly rather than as an unexplained empty cell.
# Positive infinity remains represented as "inf" in the internal CSV.
pilot_features.to_csv(
    OUTPUT_FEATURES_CSV,
    index=False,
    na_rep="NaN",
)

pilot_ground_truth.to_csv(
    OUTPUT_GROUND_TRUTH_CSV,
    index=False,
)


def calculate_sha256(file_path):
    """
    Calculate a SHA-256 checksum without loading the whole file at once.
    """
    digest = hashlib.sha256()

    with open(file_path, "rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


features_sha256 = calculate_sha256(
    OUTPUT_FEATURES_CSV
)

ground_truth_sha256 = calculate_sha256(
    OUTPUT_GROUND_TRUTH_CSV
)


manifest = {
    "artifact_status": "development_smoke_test",
    "source_dataset": SOURCE_CSV.name,
    "source_dataset_sha1_verified_in": (
        "notebooks/01_data_audit.ipynb"
    ),
    "feature_policy": str(
        FEATURE_POLICY_CSV.relative_to(
            PROJECT_ROOT
        )
    ),
    "sampling_notebook": (
        "notebooks/03_pilot_sampling.ipynb"
    ),
    "random_seed": RANDOM_SEED,
    "chunk_size": CHUNK_SIZE,
    "sampling_unit": (
        "unique_48_field_model_input_profile"
    ),
    "target_categories": TARGET_CATEGORIES,
    "profiles_per_category": (
        PROFILES_PER_CATEGORY
    ),
    "total_profiles": len(
        pilot_features
    ),
    "model_input_field_count": len(
        model_input_columns
    ),
    "nonfinite_policy": (
        "retain records and represent nonfinite "
        "states explicitly in both LLM conditions"
    ),
    "ground_truth_policy": (
        "Label and Attack stored separately and "
        "excluded from model-facing features"
    ),
    "temporal_note": (
        "Class-dependent collection-date distribution "
        "requires final-protocol sensitivity analysis"
    ),
    "outputs": {
        "features_csv": {
            "path": str(
                OUTPUT_FEATURES_CSV.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": features_sha256,
        },
        "private_ground_truth_csv": {
            "path": str(
                OUTPUT_GROUND_TRUTH_CSV.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": ground_truth_sha256,
        },
    },
    "software": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "platform": platform.platform(),
    },
}

with open(
    OUTPUT_MANIFEST_JSON,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
    )


print("Saved feature table:")
print(OUTPUT_FEATURES_CSV)

print("\nSaved private ground-truth table:")
print(OUTPUT_GROUND_TRUTH_CSV)

print("\nSaved reproducibility manifest:")
print(OUTPUT_MANIFEST_JSON)

Saved feature table:
/Users/ruiwang/Developer/compsci742-rui-pilot/data/interim/pilot_benign_dos_n200_features.csv

Saved private ground-truth table:
/Users/ruiwang/Developer/compsci742-rui-pilot/data/interim/pilot_benign_dos_n200_ground_truth_private.csv

Saved reproducibility manifest:
/Users/ruiwang/Developer/compsci742-rui-pilot/data/interim/pilot_benign_dos_n200_manifest.json


In [14]:
# ---------------------------------------------------------------------
# Verify the saved files independently from the in-memory DataFrames
# ---------------------------------------------------------------------

reloaded_features = pd.read_csv(
    OUTPUT_FEATURES_CSV
)

reloaded_ground_truth = pd.read_csv(
    OUTPUT_GROUND_TRUTH_CSV
)

with open(
    OUTPUT_MANIFEST_JSON,
    "r",
    encoding="utf-8",
) as manifest_file:
    reloaded_manifest = json.load(
        manifest_file
    )

assert reloaded_features.shape == (200, 50)
assert reloaded_ground_truth.shape == (200, 7)

assert "Label" not in reloaded_features.columns
assert "Attack" not in reloaded_features.columns

assert reloaded_features[
    "sample_id"
].tolist() == reloaded_ground_truth[
    "sample_id"
].tolist()

assert calculate_sha256(
    OUTPUT_FEATURES_CSV
) == reloaded_manifest[
    "outputs"
]["features_csv"]["sha256"]

assert calculate_sha256(
    OUTPUT_GROUND_TRUTH_CSV
) == reloaded_manifest[
    "outputs"
]["private_ground_truth_csv"]["sha256"]

saved_artifact_check = pd.Series(
    {
        "features_shape": str(
            reloaded_features.shape
        ),
        "ground_truth_shape": str(
            reloaded_ground_truth.shape
        ),
        "feature_hash_matches": True,
        "ground_truth_hash_matches": True,
        "label_absent_from_features": True,
        "attack_absent_from_features": True,
        "manifest_loaded": True,
    },
    name="value",
)

saved_artifact_check

features_shape                 (200, 50)
ground_truth_shape              (200, 7)
feature_hash_matches                True
ground_truth_hash_matches           True
label_absent_from_features          True
attack_absent_from_features         True
manifest_loaded                     True
Name: value, dtype: object

## Pilot-sampling outcome

The reproducible development sample was generated successfully from the
provider-supplied NF-UNSW-NB15-v3 source CSV.

The sampling procedure scanned 2,365,424 source records and retained 200
unique 48-field model-input profiles:

- 100 Benign profiles;
- 100 DoS profiles;
- one representative source row per retained profile;
- no duplicate model-input profiles;
- no cross-label profile conflicts; and
- no ground-truth, direct-identifier or temporal-provenance fields in the
  model-input table.

Records containing non-finite values were retained. In the selected sample,
5 Benign records and 19 DoS records contain a non-finite value in one or both
second-byte fields. These values will be represented explicitly and
consistently in both LLM input conditions rather than causing entire records
to be deleted.

The sample is a balanced development and reproducibility pilot rather than a
population-representative evaluation sample. Capture-date composition differs
between Benign and DoS traffic and remains a potential collection-context
confound requiring consideration in the final experimental protocol.

Three local interim artifacts were generated:

1. a label-free model-feature table;
2. a private ground-truth and provenance table; and
3. a reproducibility manifest containing configuration details and SHA-256
   checksums.

Reloading the saved artifacts confirmed their expected shapes, label
separation and checksum integrity.